In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split as tts
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

In [3]:
df = pd.read_csv("Cleaned_Data.csv")
df.head()

,CustomerID,Age,Annual_Income_k$,Spending_Score,Purchase_Frequency,Gender
0,1,0.856678,0.166310,-0.693726,-0.894227,0
1,2,1.747915,1.661890,-0.296783,-0.894227,1
2,3,0.171110,-0.525600,0.683898,0.892235,0
3,4,-0.788684,-0.621838,-0.156686,1.338850,0
4,5,1.130905,-0.119302,0.333655,-1.340842,0


In [7]:
X = df[["Age" , "Annual_Income_k$" , "Purchase_Frequency" , "Gender"]]
y = df["Spending_Score"]

In [ ]:
# Train Test Split : to break data on 80 (training) : 20 (testing) ration

In [8]:
X_train, X_test, y_train, y_test = tts(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train:- {X_train.shape}")
print(f"Shape of X_test:- {X_test.shape}")
print(f"Shape of y_train:- {y_train.shape}")
print(f"Shape of y_test:- {y_test.shape}")

Shape of X_train:- (80000, 4)
Shape of X_test:- (20000, 4)
Shape of y_train:- (80000,)
Shape of y_test:- (20000,)


In [ ]:
# train models : Linear , Decision tree , Random Forest , K- Means , PCA 

In [27]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    
    results.append({
        "Model": name,
        "R2 Score": round(r2, 4),
        "MAE": round(mae, 4)
    })

In [40]:
performance_df = pd.DataFrame(results)
print("Model Performance Report")
print(performance_df.to_string(index=False))

# find best model by r2 score
best_model_name = performance_df.loc[performance_df["R2 Score"].idxmax()]["Model"]
print(f"\nBest Model Found: {best_model_name}")

# save model by joblib
model_filename = "best_customer_model.joblib"
joblib.dump([best_model_name], model_filename) 
print(f"Success: Actual {best_model_name} object saved as {model_filename}")

# model load on joblib : jisse pata lag  sake ki model successful load hua hai ya nhi
loaded_model = joblib.load(model_filename)

Model Performance Report
            Model  R2 Score    MAE
Linear Regression    0.0436 0.6428
    Decision Tree   -0.1693 0.7893
    Random Forest    0.3122 0.6245

Best Model Found: Random Forest
Success: Actual Random Forest object saved as best_customer_model.joblib
Success: Model loaded back from file


In [49]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor

# 1. Data load karein
df = pd.read_csv("customer_segmentation_uncleaned_data.csv")
df = df.dropna()

# 2. Gender ko number mein badlein
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})

X = df[["Age", "Annual_Income_k$", "Purchase_Frequency", "Gender"]]
y = df["Spending_Score"]

# 4. Random Forest Train karein
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# 5. Model ko save karein
joblib.dump(model, "rf_direct_model.joblib")

Naya Random Forest model 'rf_direct_model.joblib' save ho gaya hai!


In [47]:
# Data Load and Cleaning
df.fillna(df.mean(numeric_only=True), inplace=True)
df["Gender"] = LabelEncoder().fit_transform(df["Gender"])

# Scaling
X = df[["Age", "Annual_Income_k$", "Purchase_Frequency", "Gender"]]
X_scaled = StandardScaler().fit_transform(X)

# K-Means Clustering
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# PCA (Principal Componant Analysis)
pca = PCA(n_components=2)
pca_features = pca.fit_transform(X_scaled)
df["PCA1"], df["PCA2"] = pca_features[:, 0], pca_features[:, 1]

# Model Training
X_final = df[["PCA1", "PCA2", "Cluster"]]
y = df["Spending_Score"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"R2 Score: {r2_score(y_test, preds):.4f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, preds):.4f}")

R2 Score: 0.2326
Mean Absolute Error (MAE): 0.6379
